In [1]:
import event_helpers
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd
import poi_geography_utils as poi_geog_utils
import world_cup_vars as wc_vars
from gtfs_curator_utils import utils

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH
credentials, _ = google.auth.default()

In [2]:
STADIUM_NAME = "sofi"
stop_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}stop_summary_{STADIUM_NAME}.parquet",
    storage_options = {"token": credentials.token}
)

In [3]:
stop_gdf.dtypes

schedule_name                         object
stop_id                               object
stop_name                             object
route_id_array                        object
route_type_array                      object
geometry                            geometry
daily_arrivals_weekday_non_event     float64
daily_arrivals_weekend_non_event     float64
daily_arrivals_weekday_event         float64
daily_arrivals_weekend_event         float64
change_daily_arrivals_weekday        float64
change_daily_arrivals_weekend        float64
combined_change_daily_arrivals       float64
near_route_ids                        object
special_route_ids                     object
is_route_near                           bool
is_special_route                        bool
is_stop_near                            bool
point_of_interest                     object
dtype: object

In [53]:
import create_route_and_stop_event_summary as A1

routes_df = A1.import_routes_near_poi(wc_vars.event_name, "sofi")

stop_gdf = A1.prep_fct_daily_scheduled_stops(wc_vars.event_name, wc_vars.sofi_match_times).merge(
    routes_df[["feed_key", "schedule_name"]].drop_duplicates(),
    how="inner",
)


In [54]:
stop_gdf.dtypes

service_date                  datetime64[ns]
feed_key                              object
stop_id                               object
stop_name                             object
daily_arrivals                         Int64
geometry                            geometry
arrivals_per_hour_owl                float64
arrivals_per_hour_early_am           float64
arrivals_per_hour_am_peak            float64
arrivals_per_hour_midday             float64
arrivals_per_hour_pm_peak            float64
arrivals_per_hour_evening            float64
arrivals_owl                           Int64
arrivals_early_am                      Int64
arrivals_am_peak                       Int64
arrivals_midday                        Int64
arrivals_pm_peak                       Int64
arrivals_evening                       Int64
route_id_array                        object
route_type_array                      object
event_day                               bool
day_type                              object
event_time

In [55]:
# sofi has midday and pm_peak
time_of_day = "midday"

# midday matches that are also weekday
# midday matches that are weekend


In [56]:
from typing import Literal

def grab_matches_by_day_type(
    event_time_of_day_dict: dict,
    time_of_day: Literal["early_am", "am_peak", "midday", "pm_peak", "evening", "owl"],
    day_type: Literal["weekday", "weekend"],
) -> dict:
    """
    Or create a new dict that can key into weekday or weekend?
    """
    filtered_events = {
        **{d: tod for d, tod in wc_vars.sofi_match_times.items()
          if (tod == time_of_day) and (
              pd.to_datetime(d).day_name() in ["Saturday", "Sunday"] if day_type == "weekend" 
              else pd.to_datetime(d).day_name() in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
          )}
    }
    
    return filtered_events

In [57]:
time_of_day = "midday"
day_type = "weekday"

filtered_sofi_dict = grab_matches_by_day_type(wc_vars.sofi_match_times, time_of_day, day_type)
filtered_sofi_dict

{'2026-06-18': 'midday', '2026-07-02': 'midday', '2026-07-10': 'midday'}

In [62]:
# how should comparison be done, grab those dates out and grab all the non-event dates? put those into aggregation?
arrivals_cols = [c for c in stop_gdf.columns if "arrivals_per_hour" in c]

arrivals_by_event_df = (
    event_helpers.aggregate_by_event_type(
        stop_gdf,
        group_cols=[
            "schedule_name",
            "stop_id",
            "stop_name",
            "event_day",
            "day_type",
        ],
        mean_cols=arrivals_cols
    ).query('day_type==@day_type')
    .pipe(event_helpers.merge_in_stop_geom, stop_gdf)
)

In [63]:
arrivals_by_event_df.dtypes

schedule_name                   object
stop_id                         object
stop_name                       object
route_id_array                  object
route_type_array                object
geometry                      geometry
event_day                         bool
day_type                        object
daily_arrivals                 Float64
arrivals_per_hour_owl          float64
arrivals_per_hour_early_am     float64
arrivals_per_hour_am_peak      float64
arrivals_per_hour_midday       float64
arrivals_per_hour_pm_peak      float64
arrivals_per_hour_evening      float64
n_days                           int64
dtype: object